# HIT140 Foundations of Data Science — Assignment 2
## Positional Shooting Precision
**Analytic question:** Do Forwards record a significantly higher average of Shots on Target per 90 minutes than Midfielders?

Run the notebooks in numerical order. Each notebook reads the CSV exported by the preceding stage, so the workflow remains reproducible.

# Notebook 1 — Data Wrangling
This notebook performs data wrangling on the raw player dataset. It selects relevant variables, checks and handles data-quality issues, applies the participation and position inclusion criteria, restructures position information, assigns unique player IDs, and exports the analysis-ready population for the next stage.


In [ ]:
import pandas as pd
import os

# Raw input file supplied for the assignment.
csv_path = "Player_Shots_on_Target_data.csv"

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        "Place 'Player_Shots_on_Target_data.csv' in the same folder as this notebook, then run again."
    )

df = pd.read_csv(csv_path, encoding="cp1252")

print("Raw dataset shape:", df.shape)
display(df.head())


In [ ]:
# Select only variables relevant to the analytic question.
required_columns = ["Player", "Pos", "Squad", "Age", "90s", "Sh", "SoT", "SoT/90"]

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df_clean = df[required_columns].copy()

# Check missing values before cleaning.
print("Missing values before cleaning:")
display(df_clean.isna().sum().to_frame("Missing_Count"))


In [ ]:
# Remove duplicate rows if any are present.
duplicates_before = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().copy()

# Keep players who actually appeared (more than 0 equivalent 90-minute periods).
df_clean = df_clean[df_clean["90s"] > 0].copy()

# Use the first listed position as the player's primary position.
df_clean["Primary_Position"] = df_clean["Pos"].str.split(",").str[0].str.strip()

# Keep only Forwards and Midfielders.
df_clean = df_clean[df_clean["Primary_Position"].isin(["FW", "MF"])].copy()

# Give the groups clear labels.
df_clean["Position_Group"] = df_clean["Primary_Position"].map(
    {"FW": "Forward", "MF": "Midfielder"}
)

# Remove records missing the outcome or position group.
df_clean = df_clean.dropna(subset=["SoT/90", "Position_Group"]).copy()

# Assign a reproducible unique ID after cleaning.
df_clean = df_clean.reset_index(drop=True)
df_clean["Player_ID"] = ["PL" + str(i).zfill(4) for i in range(1, len(df_clean) + 1)]

print("Duplicate rows removed:", duplicates_before)
print("Wrangled dataset shape:", df_clean.shape)
print("\nEligible players by position:")
print(df_clean["Position_Group"].value_counts())
display(df_clean.head())


In [ ]:
# Final quality checks.
print("Missing values after cleaning:")
display(df_clean.isna().sum().to_frame("Missing_Count"))

print("\nSoT/90 data type:", df_clean["SoT/90"].dtype)
print("Unique Player_ID values:", df_clean["Player_ID"].nunique())
print("Total wrangled rows:", len(df_clean))


In [ ]:
# Export the wrangled population for Notebook 2 and later population checks.
output_file = "01_wrangled_player_data.csv"
df_clean.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Exported: {output_file}")
